In [1]:
import json

input_path = r"H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\DeepSeek-R1-Distill-Qwen-7B\generated-responses_with-leakage-spans_luna_judge.jsonl"
output_path = r"H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\DeepSeek-R1-Distill-Qwen-7B\generated-responses_with-leakage-spans_luna_judge_cleaned.jsonl"

with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for line in fin:
        if not line.strip():
            continue

        data = json.loads(line)

        # Remove leakage_spans if it is empty
        if data.get("leakage_spans") == []:
            continue

        fout.write(json.dumps(data, ensure_ascii=False) + "\n")

print(f"Saved cleaned file to: {output_path}")

Saved cleaned file to: H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\DeepSeek-R1-Distill-Qwen-7B\generated-responses_with-leakage-spans_luna_judge_cleaned.jsonl


In [ ]:
{'error': {'message': "Unknown parameter: 'chat_template_kwargs'.", 'type': 'invalid_request_error', 'param': 'chat_template_kwargs', 'code': 'unknown_parameter'}}

In [3]:
from pathlib import Path
import json
import random
import shutil


src = Path(
    r"H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\DeepSeek-R1-Distill-Qwen-14B\generated-responses_with-leakage-spans.jsonl"
)

backup = src.with_suffix(src.suffix + ".bak")
test_path = src.with_name("test_" + src.name)
val_path = src.with_name("val_" + src.name)


# Create backup if it does not already exist
if not backup.exists():
    shutil.copy2(src, backup)


# Always split from the original backup so rerunning is safe
records = [
    json.loads(line)
    for line in backup.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

# Deterministic shuffle
random.Random(42).shuffle(records)


# Calculate split sizes
n = len(records)
n_test = round(n * 0.15)
n_val = round(n * 0.15)


# Create splits: 15% test, 15% validation, 70% train
splits = {
    test_path: records[:n_test],
    val_path: records[n_test:n_test + n_val],
    src: records[n_test + n_val:],
}


# Write each split
for path, rows in splits.items():
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


# Print summary
print(f"total={n}")
print(f"test={len(splits[test_path])}: {test_path}")
print(f"val={len(splits[val_path])}: {val_path}")
print(f"train={len(splits[src])}: {src}")
print(f"backup={backup}")

total=289
test=43: H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\DeepSeek-R1-Distill-Qwen-14B\test_generated-responses_with-leakage-spans.jsonl
val=43: H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\DeepSeek-R1-Distill-Qwen-14B\val_generated-responses_with-leakage-spans.jsonl
train=203: H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\DeepSeek-R1-Distill-Qwen-14B\generated-responses_with-leakage-spans.jsonl
backup=H:\Project\traffic-hope-network\method\calibrate_leakage_detector\data\DeepSeek-R1-Distill-Qwen-14B\generated-responses_with-leakage-spans.jsonl.bak


In [2]:
def harmonic_mean(a, b):
    """Calculate the harmonic mean of two numbers."""
    if a <= 0 or b <= 0:
        raise ValueError("Both numbers must be positive.")
    return 2 * (a * b) / (a + b)

In [3]:
harmonic_mean(0.9, 0.1)

0.18000000000000002

In [4]:
harmonic_mean(0.5, 0.5)

0.5

In [6]:
harmonic_mean(0.8, 0.4)

0.5333333333333333

In [ ]:
import os, re, json, asyncio
from pathlib import Path

from datasets import load_dataset
from openai import AsyncOpenAI
from tqdm.auto import tqdm


# =====================
# Config
# =====================
MODEL = "Qwen/Qwen3-4B"
BASE_URL = "http://localhost:8000/v1"
API_KEY = "EMPTY"

N_SAMPLES = 100
MAX_CONCURRENCY = 8
MAX_NEW_TOKENS_NATIVE = 2048
MAX_NEW_TOKENS_REWRITE = 1024

OUT_DIR = Path("method/calibrate_leakage_detector/data/reasoning_trace_compare")
NATIVE_PATH = OUT_DIR / "native_reasoning_traces.jsonl"
NONNATIVE_PATH = OUT_DIR / "nonnative_reasoning_traces.jsonl"

OUT_DIR.mkdir(parents=True, exist_ok=True)

client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)


# =====================
# Helpers
# =====================
def extract_think_and_answer(text):
    m = re.search(r"<think>(.*?)</think>", text, flags=re.S)
    if not m:
        return "", text.strip()

    native_think = m.group(1).strip()
    final_answer = text[m.end():].strip()
    return native_think, final_answer


async def complete(messages, max_tokens, enable_thinking=None):
    kwargs = {
        "model": MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": 0.0,
    }

    # For Qwen-style vLLM servers. Remove this block if your endpoint rejects it.
    if enable_thinking is not None:
        kwargs["extra_body"] = {
            "chat_template_kwargs": {"enable_thinking": enable_thinking}
        }

    r = await client.chat.completions.create(**kwargs)
    return r.choices[0].message.content or ""


def write_jsonl(path, rows):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


# =====================
# Prompts
# =====================
def native_prompt(problem):
    return [
        {
            "role": "user",
            "content": f"Solve the following math problem.\n\nProblem:\n{problem}",
        }
    ]


def rewrite_prompt(problem, final_answer):
    return [
        {
            "role": "user",
            "content": (
                "Rewrite the reasoning for the following math problem as a clean, concise "
                "step-by-step solution. Do not mention that you are rewriting. Do not use "
                "<think> tags.\n\n"
                f"Problem:\n{problem}\n\n"
                f"Final answer to explain:\n{final_answer}"
            ),
        }
    ]


# =====================
# Main
# =====================
async def process_one(i, row, sem):
    problem = row["question"]
    gold_answer = row["answer"]

    async with sem:
        native_response = await complete(
            native_prompt(problem),
            max_tokens=MAX_NEW_TOKENS_NATIVE,
            enable_thinking=True,
        )

    native_trace, final_response = extract_think_and_answer(native_response)

    async with sem:
        rewritten_trace = await complete(
            rewrite_prompt(problem, final_response),
            max_tokens=MAX_NEW_TOKENS_REWRITE,
            enable_thinking=False,
        )

    native_row = {
        "sample_id": i,
        "problem": problem,
        "gold_answer": gold_answer,
        "native_think": native_trace,
        "final_response": final_response,
        "raw_response": native_response,
    }

    nonnative_row = {
        "sample_id": i,
        "problem": problem,
        "gold_answer": gold_answer,
        "nonnative_think": rewritten_trace.strip(),
        "source_final_response": final_response,
    }

    return native_row, nonnative_row


async def main():
    data = load_dataset("gsm8k", "main", split=f"test[:{N_SAMPLES}]")
    sem = asyncio.Semaphore(MAX_CONCURRENCY)

    tasks = [process_one(i, row, sem) for i, row in enumerate(data, 1)]

    native_rows, nonnative_rows = [], []
    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
        native_row, nonnative_row = await coro
        native_rows.append(native_row)
        nonnative_rows.append(nonnative_row)

    native_rows.sort(key=lambda x: x["sample_id"])
    nonnative_rows.sort(key=lambda x: x["sample_id"])

    write_jsonl(NATIVE_PATH, native_rows)
    write_jsonl(NONNATIVE_PATH, nonnative_rows)

    print(f"Saved native traces to: {NATIVE_PATH}")
    print(f"Saved nonnative traces to: {NONNATIVE_PATH}")


await main()